# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Classification, deployed as ranking. The real decision is *which pages should an editor refresh first?* — not "predict decline" as a party trick. So I frame it as binary classification: will this page's search impressions fall more than 20% in the last 30 days vs the prior 30 days? That's `is_declining_label`. Then I turn the model's probability into a priority score so the editor gets a ranked queue. The model is a classifier; the ranking is just the deployment layer.

In [1]:
# Section 1 — task type
# Classification, deployed as ranking.
# Real decision: which pages should an editor refresh first?
# We predict is_declining_label (yes/no) and surface a ranked queue on top.

## 2. Target or proxy

Target: `is_declining_label` (1 = declining, 0 = not). It comes from an *observed* outcome — Google Search Console impression counts, last 30 days vs the previous 30 days (`trend_direction == "down"`). That's measured signal, not a human opinion or a hand-written rule, so the model learns the real world, not a rule.

Leakage guard (critical): the label is built *from* `trend_direction` and `trend_pct`, so those two columns — and the IDs — can never be features. Notebook 02 shows exactly what happens if you leak them in.

In [2]:
# One-paragraph frame:
# For editors deciding which pages to refresh, we build a classification model from
# historical search + engagement data, predicting is_declining_label
# (trend_direction == "down"), measured by client-holdout Precision@50 and ROC-AUC.
# A wrong call costs wasted editor hours (FP) or a missed decline that compounds (FN).
# A plain rule isn't enough because decline is multi-signal (position drift + volume
# change + engagement erosion) -- no single threshold catches them all.

## 3. Success metric

One metric I can defend: **Precision@50 on a client-holdout.** Of the 50 pages the system says to fix first, what fraction are genuinely declining? Higher = fewer wasted editor hours.

The hand-rule baseline scores about **0.24** on the client-holdout; the model gets about **0.74** — roughly a **3× lift** (the stable claim is the lift, not the third decimal). For *picking* the model I use **ROC-AUC**: it's threshold-free, so it's fair at the ~0.54 base rate where raw accuracy lies.

In [3]:
# Success metric: Precision@50 on a client-holdout.
# Floor (hand-rule baseline) ~0.24; model ~0.74 (~3x lift); model-selection = ROC-AUC.
# Real numbers are printed in the data-look cell below (Section 4).

## 4. The unit of analysis, as a real dataframe

One row = one content item (page). 30,000 pseudonymized pages across 32 clients, with trailing-90-day search and analytics metrics. The code cell loads it and shows the shape, the derived label rate, the metric numbers, and a few rows — so the grain is real before we model on it.

In [4]:
# Section 4 — the unit of analysis, as a real dataframe (the one real code cell, like w01)
from pathlib import Path
import json as _json
import pandas as pd

# find the anonymized CSV whether we run locally, from work/notebooks/, or as work/w02.ipynb
def find_raw_csv():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        cand = parent / "data" / "raw" / "content_refresh_anonymized.csv"
        if cand.exists():
            return cand
    return Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(find_raw_csv())
df.columns = df.columns.str.strip()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns, {df['client_id'].nunique()} clients")
print()

# Label rate (base rate)
base_rate = df["is_declining_label"].mean()
print(f"Declining (is_declining_label=1): {base_rate:.3f}  -> a balanced binary problem")
print()

# Metric numbers from the reference pipeline (outputs/model_results.json), like w01's real numbers.
_mr_path = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    cand = parent / "outputs" / "model_results.json"
    if cand.exists():
        _mr_path = cand
        break
if _mr_path is not None:
    _mr = _json.loads(_mr_path.read_text())
    baseline_p50 = _mr["baseline"]["baseline_precision_at_50"]
    model_p50 = _mr["models"][_mr["best_model"]["name"]]["precision_at_50"]
    src = "reference pipeline"
else:
    baseline_p50, model_p50 = 0.240, 0.740
    src = "documented (README/GUIDE FAQ)"
print(f"Precision@50 floor (hand-rule baseline): {baseline_p50:.3f}  [{src}]")
print(f"Precision@50 (best model):              {model_p50:.3f}")
print(f"Lift over baseline:                     {model_p50 / baseline_p50:.1f}x")
print()

# A few real rows to confirm the grain (pseudonymized ids only, no client names)
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position",
        "days_since_last_update", "trend_direction", "is_declining_label"]
print(df[cols].head(3).to_string(index=False))

Shape: 30,000 rows, 45 columns, 32 clients

Declining (is_declining_label=1): 0.542  -> a balanced binary problem

Precision@50 floor (hand-rule baseline): 0.240  [reference pipeline]
Precision@50 (best model):              0.740
Lift over baseline:                     3.1x

          content_id         client_id  impressions_90d  clicks_90d  avg_position  days_since_last_update trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc             3803          29          10.6                      20            down                   1
content_a1fb4e703a9e client_4e07408562            15320           7          20.3                      25            down                   1
content_9aa793d4d895 client_7f2253d7e2            12581          11          36.5                      20            down                   1


## 5. Why ML beats a fixed rule here

Decline is too messy for an if-statement. No single feature separates declining from not — the strongest single-feature correlation with decline is only about **0.19**. Instead it's a tangle of weak, partly non-linear effects: visibility, freshness, position, engagement, AI-referral traffic, all contributing a little and interacting. A model learns that blend from data; a hand rule would need dozens of brittle thresholds that drift as Google changes. That's exactly where ML earns its place.

In [5]:
# Why ML beats a rule: no single feature separates the classes (max |corr| ~0.19).
# The pattern is many weak, entangled, non-linear signals; a model learns the blend,
# a hand rule would need dozens of brittle thresholds that drift over time.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/w02` — then submit your repo URL on the card. Done.